# Build a Resume AI Assistant

This is the original exploration notebook behind the **Resume AI Assistant** project. It:

1. Warms up with Pydantic models and OpenAI structured outputs.
2. Defines `openai_generate` / `claude_generate` helpers for OpenAI and Anthropic Claude.
3. Analyzes a resume against a job description (key requirements, relevant experience, gaps, strengths).
4. Rewrites the resume for the job and highlights the changes.
5. Generates a matching cover letter, and chains everything in `run_resume_rocket`.

The production version of this logic lives in [`app/`](../app) and is served by the Streamlit app (`streamlit run app/app.py`).

**Setup:** copy `../.env.example` to `../.env` and set `OPENAI_API_KEY` / `ANTHROPIC_API_KEY`. Never commit your `.env` file or API keys.

**Privacy:** the example resume and job description are fictional (loaded from `../examples/`). Saved outputs have been cleared; clear them again before committing if you run it with real data.

In [ ]:
# %pip install -r ../requirements.txt
from pydantic import BaseModel

In [ ]:
class User(BaseModel):
    name:str
    age:int
    email:str

In [ ]:
user = User(name = "John Doe", age = 30, email = "john.doe@example.com")
print(user.model_dump_json())

In [ ]:
# Dependencies (anthropic, openai, python-dotenv, ...) are listed in ../requirements.txt

In [ ]:
import os
import anthropic
from openai import OpenAI
from dotenv import load_dotenv
import json

from typing import List, Dict, Union, Optional, Any
from IPython.display import display, Markdown

print("Libraries imported successfully!")

# API keys are read from environment variables (or a local, git-ignored .env file).
# Never paste a real key into this notebook.
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

if not openai_api_key or not anthropic_api_key:
    print("Warning: set OPENAI_API_KEY and ANTHROPIC_API_KEY in your environment or .env file.")

openai_client = OpenAI(api_key=openai_api_key)
claude_client = anthropic.Anthropic(api_key=anthropic_api_key)

In [ ]:
class scientist(BaseModel):
  name: str
  field: str
  known_for: list[str]
  birth_year: int

In [ ]:
prompt = """
Give me a JSON object with details about a famous scientist.
Include the following fields: name, field, known_for, and birth_year.
"""

response = openai_client.chat.completions.parse(model = "gpt-4o",
                                                 messages = [{"role": "user", 
                                                 "content": prompt}],
    temperature = 0,
    response_format = scientist,
    max_tokens = 300)

In [ ]:
response.choices[0].message.content

In [ ]:
json.loads(response.choices[0].message.content)

In [ ]:
response.choices[0].message.parsed

In [ ]:
def print_markdown(text):
    display(Markdown(text))

In [ ]:
from pathlib import Path

# Fictional demo data. Replace with your own resume locally - do not commit personal data.
resume_text = Path("../examples/sample_resume.md").read_text(encoding="utf-8")

In [ ]:
# Fictional demo job description.
job_description_text = Path("../examples/sample_job_description.md").read_text(encoding="utf-8")

In [ ]:
print_markdown("**--- Original Resume ---**")
print_markdown(resume_text)

In [ ]:
print_markdown("\n**--- Target Job Description ---**")
print_markdown(job_description_text)

In [ ]:
def openai_generate(prompt: str,
                    model:str = "gpt-4o",
                    temperature: float=0.7,
                    max_tokens: int=1500,
                    response_format: Optional[dict]=None) -> str | dict:
    

    try:
        if not response_format:
            response = openai_client.chat.completions.create(
                model = model,
                messages=[
                    {"role":"system",
                     "content": "You are a helpful assistant specializing in resume writing and career advice.",},
                     {"role":"user", "content":prompt}

                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        
        else:
            completion = openai_client.chat.completions.parse(
                model = model,
                messages=[
                    {
                        "role":"system",
                        "content":"You are a helpful assistant specializing in resume writing and career advice."
                    },
                    {"role":"user", "content":prompt},
                ],
                temperature=temperature,
                response_format=response_format
            )
            
            return completion.choices[0].message.parsed
    
    except Exception as e:
        return f"Error generating text: {e}"

In [ ]:
prompt = f"""
Context:
You are a professional resume writer helping a candidate tailor their resume for a specific job opportunity. The resume and job description are provided below.

Instruction:
Enhance the resume to make it more impactful. Focus on:
- Highlighting relevant skills and achievements.
- Using strong action verbs and quantifiable results where possible.
- Rewriting vague or generic bullet points to be specific and results-driven.
- Emphasizing experience and skills most relevant to the job description.
- Reorganizing sections if necessary to better match the job requirements.

Resume:
{resume_text}

Output:
Provide a revised and improved version of the resume that is well-formatted. Only return the updated resume.
"""

In [ ]:
openai_output = openai_generate(prompt, temperature=0.7)

print_markdown("#### OpenAI Response:")
print_markdown(openai_output)

In [ ]:
from typing import Optional, Any
from pydantic import BaseModel
import json

def claude_generate(prompt: str,
                    model: str = "claude-sonnet-4-6",
                    temperature: float = 0.7,  # kept for API symmetry; not supported by anthropic>=1.0
                    max_tokens: int = 4000,
                    response_format: Optional[Any] = None) -> str | Any:

    system_prompt = "You are a helpful assistant specializing in resume writing and career advice."

    try:
        if not response_format:
            response = claude_client.messages.create(
                model=model,
                max_tokens=max_tokens,
                system=system_prompt,
                messages=[{"role": "user", "content": prompt}]
            )

            return response.content[0].text

        else:
            # Native structured outputs: the SDK validates the reply against the Pydantic model.
            response = claude_client.messages.parse(
                model=model,
                max_tokens=max_tokens,
                system=system_prompt,
                messages=[{"role": "user", "content": prompt}],
                output_format=response_format
            )

            return response.parsed_output

    except Exception as e:
        return f"Error generating text: {e}"

In [ ]:
def cloude_generate2(prompt: str, model: str = "claude-sonnet-4-6", temperature: float = 0.7, max_output_tokens: int = 1500) -> str:

    try:
        response = claude_client.messages.create(
                model=model,
                max_tokens=max_output_tokens,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
        # Return just the text content from the response
        return response.content[0].text
    except Exception as e:
        # Error handling to prevent crashes
        return f"Error generating text: {e}"

In [ ]:
cloude_output = cloude_generate2(prompt, temperature = 0.7)

print_markdown("#### Cloude Response:")
print_markdown(cloude_output)

In [ ]:
claude_client.models.list()

In [ ]:
def analyze_resume_against_job_description(job_description_text: str, resume_text: str, model: str = "openai") -> str:
   
    prompt = f"""
    Context:
    You are a career advisor and resume expert. Your task is to analyze a candidate's resume against a specific job description to assess alignment and identify areas for improvement.

    Instruction:
    Review the provided Job Description and Resume. Identify key skills, experiences, and qualifications in the Job Description and compare them to what's present in the Resume. Provide a structured analysis with the following sections:
    1. **Key Requirements from Job Description:** List the main skills, experiences, and qualifications sought by the employer.
    2. **Relevant Experience in Resume:** List the skills and experiences from the resume that match or align closely with the job requirements.
    3. **Gaps/Mismatches:** Identify important skills or qualifications from the Job Description that are missing, unclear, or underrepresented in the Resume.
    4. **Potential Strengths:** Highlight any valuable skills, experiences, or accomplishments in the resume that are not explicitly requested in the job description but could strengthen the application.

    Job Description:

    {job_description_text}

    Resume:

    {resume_text}

    Output:
    Return a clear, structured comparison with the four sections outlined above.
    """

    if model == "openai":
        gap_analysis = openai_generate(prompt, temperature=0.7)
    elif model == "claude":
        gap_analysis = claude_generate(prompt, temperature=0.5)
    else:
        raise ValueError(f"Invalid model: {model}")

    return gap_analysis



In [ ]:
gap_analysis_openai = analyze_resume_against_job_description(job_description_text, 
                                                             resume_text, 
                                                             model = "openai")

print_markdown("#### Response:")
print_markdown(gap_analysis_openai)

In [ ]:
class ResumeOutput(BaseModel):
    updated_resume: str
    diff_markdown: str

def generate_resume(job_description_text:str, resume_text:str, gap_analysis_openai:str, model:str = "openai") -> dict:

    prompt = (
        """
    ### Context:
    You are an expert resume writer and editor. Your goal is to rewrite the original resume to match the target job description, using the provided tailoring suggestions and analysis.

    ---

    ### Instruction:
    1. Rewrite the entire resume to best match the **Target Job Description** and **Gap Analysis to the Job Description**.
    2. Improve clarity, add job-relevant keywords, and quantify achievements.
    3. Address the gaps identified in the analysis honestly by:
       - Surfacing relevant skills and technologies that are already evidenced in the original resume
       - Reframing experience to highlight relevant accomplishments
       - Strengthening sections that were identified as weak in the analysis
       - Never inventing experience, employers, job titles, dates, metrics, skills or certifications
    4. Prioritize addressing the most critical gaps first
    5. Incorporate industry-specific terminology from the job description
    6. Ensure all quantifiable achievements are properly highlighted with metrics
    7. Return two versions of the resume:
        - `updated_resume`: The final rewritten resume (as plain text)
        - `diff_markdown`: A version of the resume with inline highlights using color:
            - Additions or rewritten content should be **green**:  
            `<span style="color:green">your added or changed text</span>`
            - Removed content should be **red and struck through**:  
            `<span style="color:red;text-decoration:line-through">removed text</span>`
            - Leave unchanged lines unmarked.
        - Keep all section headers and formatting consistent with the original resume.

    ---

    ### Output Format:

    ```json
    {
    "updated_resume": "<full rewritten resume as plain text>",
    "diff_markdown": "<HTML-colored version of the resume highlighting additions and deletions>"
    }
    ```
    ---
    ### Input:

    **Original Resume:**

    """
        + resume_text
        + """


    **Target Job Description:**

    """
        + job_description_text
        + """


    **Analysis of Resume vs. Job Description:**

    """
        + gap_analysis_openai
    )

    if model == "openai":
        updated_resume_json = openai_generate(prompt, temperature=0.7, response_format= ResumeOutput)
    elif model == "claude":
        updated_resume_json = claude_generate(prompt, temperature=0.5, response_format=ResumeOutput)
    else:
        raise ValueError(f"Invalid model: {model}")
    
    return updated_resume_json

In [ ]:
updated_resume_json = generate_resume(job_description_text, resume_text, gap_analysis_openai, model="openai")

print_markdown(updated_resume_json.updated_resume)

In [ ]:
print_markdown(updated_resume_json.diff_markdown)

In [ ]:
class CoverLetterOutput(BaseModel):
    cover_letter: str


def generate_cover_letter(job_description_text: str, updated_resume: str, model: str = "openai") -> dict:

    prompt = (
        """
    ### Context:
    You are a professional career coach and expert cover letter writer.

    ---

    ### Instruction:
    Write a compelling, personalized cover letter based on the **Updated Resume** and the **Target Job Description**. The letter should:
    1. Be addressed generically (e.g., "Dear Hiring Manager")
    2. Be no longer than 4 paragraphs
    3. Highlight key achievements and experiences from the updated resume
    4. Align with the responsibilities and qualifications in the job description
    5. Reflect the applicant's enthusiasm and fit for the role
    6. End with a confident and polite closing statement
    7. Only mention experience that appears in the resume; never invent facts

    ---

    ### Output Format (JSON):
    ```json
    {
    "cover_letter": "<final cover letter text>"
    }
    ```
    ---

    ### Input:

    **Updated Resume:**

    """
        + updated_resume
        + """
    **Target Job Description:**

    """
        + job_description_text
    )

    # Depending on the selected model, call the appropriate function to generate the cover letter.
    if model == "openai":
        # Get response from OpenAI API
        updated_cover_letter = openai_generate(prompt, temperature=0.7, response_format=CoverLetterOutput)
    elif model == "claude":
        updated_cover_letter = claude_generate(prompt, temperature=0.5, response_format=CoverLetterOutput)
    else:
        # Raise an error if an invalid model name is provided.
        raise ValueError(f"Invalid model: {model}")

    # Return the generated cover letter as a dictionary.
    return updated_cover_letter

In [ ]:
updated_cover_letter = generate_cover_letter(job_description_text, updated_resume_json.updated_resume, model="openai")

print_markdown(updated_cover_letter.cover_letter)

In [ ]:
def run_resume_rocket(resume_text: str, job_description_text: str) -> tuple[str, str]:

    gap_analysis_openai = analyze_resume_against_job_description(job_description_text, 
                                                                 resume_text, 
                                                                 model="openai")

    print_markdown(gap_analysis_openai)

    print("\n--------------------------------")
    print("--------------------------------\n")

    updated_resume_json = generate_resume(job_description_text, 
                                          resume_text, 
                                          gap_analysis_openai, 
                                          model = "openai")

    print_markdown(updated_resume_json.diff_markdown)

    print("\n--------------------------------")
    print("--------------------------------\n")

    print_markdown(updated_resume_json.updated_resume)

    print("\n--------------------------------")
    print("--------------------------------\n")

    updated_cover_letter = generate_cover_letter(
        job_description_text, updated_resume_json.updated_resume, model="openai"
    )

    print_markdown(updated_cover_letter.cover_letter)

    print("\n--------------------------------")
    print("--------------------------------\n")

    return updated_resume_json.updated_resume, updated_cover_letter.cover_letter

In [ ]:
resume, cover_letter = run_resume_rocket(resume_text, job_description_text)